In [26]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from typing import TypedDict , Annotated,Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator
from langchain_core.messages import SystemMessage, HumanMessage,BaseMessage
from langgraph.checkpoint.memory import MemorySaver

In [9]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):
    message:Annotated[list[BaseMessage],add_messages]

In [18]:
llm=ChatGroq(model="llama-3.3-70b-versatile")
def chat_node(state:ChatState):
    # Initialize the LLM
 
    # Create messages for the LLM
    messages = state['message']
    
    #send to llm
    response=llm.invoke(messages)
    
    #response store state
    return {'message':[response]}

In [27]:
checkpointer=MemorySaver()
graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)



In [23]:
initial_state={
    'message':[HumanMessage(content="what is the capital of India?")]
}
chatbot.invoke(initial_state)

{'message': [HumanMessage(content='what is the capital of India?', additional_kwargs={}, response_metadata={}, id='323ee91a-619d-43f6-bb9d-c0d2b9a16669'),
  AIMessage(content='The capital of India is New Delhi.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.010197097, 'completion_tokens_details': None, 'prompt_time': 0.003726764, 'prompt_tokens_details': None, 'queue_time': 0.048901215, 'total_time': 0.013923861}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d3adf-2da3-78b2-8331-5dc6b4e21b74-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}

In [29]:
thread_id='1'
while True:
    user_message=input('Type here:')
    print(f'User: {user_message}')
    if user_message.strip().lower() in ['exit','quit','bye']:
        break
    config={'configurable':{'thread_id': thread_id}}
    response=chatbot.invoke({'message':[HumanMessage(content=user_message)]},config=config)
    print(f'Assistant: {response["message"][-1].content}')

User: Hi , My name is Sushil
Assistant: Hello Sushil, it's nice to meet you. Is there something I can help you with or would you like to chat?
User: what is my name
Assistant: Your name is Sushil.
User: what is captial of India
Assistant: The capital of India is New Delhi.
User: can you add 10 to 100
Assistant: 100 + 10 = 110
User: can you mutiply by 5 from previous result
Assistant: The previous result was 110. Multiplying 110 by 5 gives:

110 * 5 = 550
User: exit
